In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import wandb
from pathlib import Path

from models.moodtiny import create_moodtiny, create_moodtiny_efficientnet, create_moodtiny_with_aggregation
from utils.datasets import EmotionDataset, create_dataloaders
from utils.metrics import MetricsTracker, count_parameters, compute_inference_speed
from utils.training import Trainer, create_optimizer, create_scheduler
from utils.visualization import plot_training_curves, plot_class_distribution, visualize_predictions
from utils.export import export_to_onnx

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Configuration

In [ ]:
# Load training config
with open('../configs/moodtiny.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration:")
print(yaml.dump(config, default_flow_style=False))

## 2. Prepare Dataset

In [ ]:
# Create datasets
train_dataset = EmotionDataset(
    root=config['dataset']['root'],
    split='train',
    img_size=config['model']['img_size'],
    emotions=config['classes'],
    blur_background=config['privacy']['blur_background']
)

val_dataset = EmotionDataset(
    root=config['dataset']['root'],
    split='val',
    img_size=config['model']['img_size'],
    emotions=config['classes'],
    blur_background=config['privacy']['blur_background']
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Create dataloaders
train_loader = create_dataloaders(
    train_dataset,
    batch_size=config['training']['batch_size'],
    num_workers=config['dataset']['num_workers'],
    split='train'
)

val_loader = create_dataloaders(
    val_dataset,
    batch_size=config['validation']['batch_size'],
    num_workers=config['dataset']['num_workers'],
    split='val'
)

In [ ]:
# Visualize class distribution
plot_class_distribution(
    dataset=train_dataset,
    class_names=config['classes'],
    save_path='moodtiny_class_distribution.png'
)
plt.show()

In [ ]:
# Visualize sample faces
sample_img, sample_label = train_dataset[0]
print(f"Image shape: {sample_img.shape}")  # (3, H, W)
print(f"Label: {config['classes'][sample_label]}")

# Display sample
plt.figure(figsize=(5, 5))
img_display = sample_img.permute(1, 2, 0).numpy()
img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min())
plt.imshow(img_display)
plt.title(f"Emotion: {config['classes'][sample_label]}")
plt.axis('off')
plt.tight_layout()
plt.savefig('moodtiny_sample.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Create Model

In [ ]:
# Create model
model_variant = config['model'].get('variant', 'mobilenet')  # 'mobilenet', 'efficientnet', or 'with_aggregation'
num_classes = config['model']['num_classes']
pretrained = config['model'].get('pretrained', True)

if model_variant == 'efficientnet':
    model = create_moodtiny_efficientnet(num_classes=num_classes, pretrained=pretrained)
    print("Using MoodTiny with EfficientNet-Lite0 backbone")
elif model_variant == 'with_aggregation':
    model = create_moodtiny_with_aggregation(num_classes=num_classes, pretrained=pretrained)
    print("Using MoodTiny with Crowd Aggregation")
else:
    model = create_moodtiny(num_classes=num_classes, pretrained=pretrained)
    print("Using MoodTiny with MobileNetV3-Small backbone")

model = model.to(device)

# Model info
param_info = count_parameters(model)
print(f"\nModel Parameters:")
print(f"  Total: {param_info['total_params']:,}")
print(f"  Trainable: {param_info['trainable_params']:,}")

# Test forward pass
dummy_input = torch.randn(2, 3, 112, 112).to(device)
dummy_output = model(dummy_input)
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {dummy_output.shape}")

## 4. Setup Training

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss(
    label_smoothing=config['training']['loss'].get('label_smoothing', 0.1)
)

# Optimizer
optimizer = create_optimizer(
    model,
    optimizer_type=config['training']['optimizer']['type'],
    lr=config['training']['optimizer']['lr'],
    weight_decay=config['training']['optimizer']['weight_decay']
)

# Scheduler
scheduler = create_scheduler(
    optimizer,
    scheduler_type=config['training']['scheduler']['type'],
    num_epochs=config['training']['num_epochs']
)

# Metrics tracker
metrics_tracker = MetricsTracker(
    num_classes=config['model']['num_classes'],
    class_names=config['classes']
)

print("Training setup complete!")

## 5. Initialize Weights & Biases (Optional)

In [ ]:
# Initialize W&B
use_wandb = config['wandb']['enabled']

if use_wandb:
    wandb.init(
        project=config['wandb']['project'],
        entity=config['wandb']['entity'],
        name=config['wandb']['name'],
        tags=config['wandb']['tags'],
        config=config
    )
    wandb.watch(model, log='all', log_freq=100)
    print("W&B initialized!")
else:
    print("W&B disabled")

## 6. Train Model

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    use_wandb=use_wandb
)

# Train
trainer.fit(
    num_epochs=config['training']['num_epochs'],
    metric_fn=metrics_tracker,
    save_best=config['checkpoints']['save_best'],
    checkpoint_dir=config['checkpoints']['save_dir']
)

## 7. Visualize Results

In [ ]:
# Plot training curves
train_losses = trainer.train_losses
val_losses = trainer.val_losses
train_accs = [m['accuracy'] for m in trainer.train_metrics]
val_accs = [m['accuracy'] for m in trainer.val_metrics]

plot_training_curves(
    train_losses=train_losses,
    val_losses=val_losses,
    train_accs=train_accs,
    val_accs=val_accs,
    save_path='moodtiny_training_curves.png'
)
plt.show()

In [ ]:
# Load best model
checkpoint_path = Path(config['checkpoints']['save_dir']) / 'best_model.pth'
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Best model (Epoch {checkpoint['epoch']}):")
print(f"  Train Loss: {checkpoint['train_loss']:.4f}")
print(f"  Val Loss: {checkpoint['val_loss']:.4f}")

In [ ]:
# Evaluate on validation set
metrics_tracker.reset()
val_results = trainer.validate(metric_fn=metrics_tracker)

print("\nValidation Results:")
print(f"  Accuracy: {val_results['accuracy']:.4f}")
print(f"  Precision: {val_results['precision_macro']:.4f}")
print(f"  Recall: {val_results['recall_macro']:.4f}")
print(f"  F1-Score: {val_results['f1_macro']:.4f}")

# Per-class metrics
print("\nPer-Class F1 Scores:")
for class_name in config['classes']:
    f1 = val_results.get(f'f1_{class_name}', 0)
    print(f"  {class_name}: {f1:.4f}")

In [ ]:
# Confusion matrix
metrics_tracker.plot_confusion_matrix(save_path='moodtiny_confusion_matrix.png')
plt.show()

## 8. Benchmark Inference Speed

In [ ]:
# Benchmark PyTorch model
benchmark_results = compute_inference_speed(
    model=model,
    input_shape=(1, 3, 112, 112),
    device=device,
    num_iterations=1000
)

print("\nInference Speed (PyTorch):")
print(f"  Avg time: {benchmark_results['avg_inference_ms']:.2f} ms")
print(f"  Throughput: {benchmark_results['fps']:.1f} FPS")

## 9. Export to ONNX

In [ ]:
# Export to ONNX
onnx_output_dir = '../../services/models/moodtiny/weights'
Path(onnx_output_dir).mkdir(parents=True, exist_ok=True)

model_name = f'moodtiny_{model_variant}.onnx'
onnx_path = str(Path(onnx_output_dir) / model_name)

success = export_to_onnx(
    model=model,
    input_shape=(1, 3, 112, 112),
    output_path=onnx_path,
    opset_version=14,
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    input_names=['input'],
    output_names=['output']
)

if success:
    print(f"\n✓ ONNX model exported to: {onnx_path}")
else:
    print("\n✗ ONNX export failed")

In [ ]:
# Validate ONNX model
from utils.export import validate_onnx_model, benchmark_onnx_model

validation_success = validate_onnx_model(
    onnx_path=onnx_path,
    pytorch_model=model,
    input_shape=(1, 3, 112, 112),
    tolerance=1e-5
)

if validation_success:
    print("\n✓ ONNX model validation passed")

# Benchmark ONNX Runtime
onnx_benchmark = benchmark_onnx_model(
    onnx_path=onnx_path,
    input_shape=(1, 3, 112, 112),
    num_iterations=1000,
    provider='CPUExecutionProvider'
)

print("\nInference Speed Comparison:")
print(f"  PyTorch: {benchmark_results['avg_inference_ms']:.2f} ms")
print(f"  ONNX Runtime: {onnx_benchmark['avg_inference_ms']:.2f} ms")
speedup = benchmark_results['avg_inference_ms'] / onnx_benchmark['avg_inference_ms']
print(f"  Speedup: {speedup:.2f}x")

## 10. Test on Sample Data

In [ ]:
# Test inference
model.eval()

# Get a batch from validation set
test_samples, test_labels = next(iter(val_loader))
test_samples = test_samples.to(device)

# Predict
with torch.no_grad():
    outputs = model(test_samples)
    probs = torch.softmax(outputs, dim=1)
    preds = torch.argmax(probs, dim=1)

# Display results
print("\nSample Predictions:")
for i in range(min(5, len(test_samples))):
    pred_class = config['classes'][preds[i]]
    true_class = config['classes'][test_labels[i]]
    confidence = probs[i, preds[i]].item()
    
    status = "✓" if pred_class == true_class else "✗"
    print(f"  {status} Predicted: {pred_class} ({confidence:.2%}) | True: {true_class}")

In [ ]:
# Visualize predictions on sample images
visualize_predictions(
    model=model,
    dataset=val_dataset,
    class_names=config['classes'],
    num_samples=9,
    device=device,
    save_path='moodtiny_predictions.png'
)
plt.show()

## 11. Test Crowd Mood Aggregation (if applicable)

In [ ]:
# Test crowd mood aggregation (if using aggregation variant)
if model_variant == 'with_aggregation':
    # Simulate multiple faces in a scene
    crowd_samples, _ = next(iter(val_loader))
    crowd_samples = crowd_samples[:8].to(device)  # Take 8 faces as a "crowd"
    
    with torch.no_grad():
        # Get individual predictions
        outputs = model(crowd_samples)
        probs = torch.softmax(outputs, dim=1)
        
        # Average crowd mood (simple aggregation)
        crowd_mood = probs.mean(dim=0)
        dominant_emotion = torch.argmax(crowd_mood)
        
    print("\nCrowd Mood Analysis:")
    print(f"  Number of faces: {len(crowd_samples)}")
    print(f"  Dominant emotion: {config['classes'][dominant_emotion]}")
    print(f"  Confidence: {crowd_mood[dominant_emotion]:.2%}")
    print("\n  Emotion distribution:")
    for i, emotion in enumerate(config['classes']):
        print(f"    {emotion}: {crowd_mood[i]:.2%}")
else:
    print("Crowd aggregation not enabled for this model variant")

## Summary

✅ **Training Complete!**

**Next Steps:**
1. Deploy model: `python ../scripts/deploy_models.py --model moodtiny`
2. Restart services: `docker-compose restart moodtiny`
3. Monitor inference: `docker-compose logs -f moodtiny`

**Model Artifacts:**
- Checkpoint: `./checkpoints/moodtiny/best_model.pth`
- ONNX: `../../services/models/moodtiny/weights/moodtiny_*.onnx`
- Metrics: Logged to W&B (if enabled)

**Performance:**
- Accuracy: Check val_results above
- Inference: Check onnx_benchmark above
- Privacy: Background blurring enabled, no face storage

**Privacy Features:**
- ✓ Background blurring for face privacy
- ✓ Crowd mood aggregation (if using aggregation variant)
- ✓ Emotion vectors only - no face image storage
- ✓ Lightweight model for edge deployment